

# Module 4: Case Study - Sentiment Analysis of Customer Feedback

### Objective: Develop a PyTorch model to analyze customer feedback and classify it into positive, neutral, or negative sentiments. This project aims to leverage customer feedback to improve business strategies.

### Step-by-Step Guide

---

### Real-World Project: Part 1 - Dataset Creation (10 mins)

**Step 1 and 2: Import Required Libraries and import the dataset**














In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel


# Load the dataset from the provided CSV file
df = pd.read_csv('/mnt/data/sentiment-analysis.csv')

# Display the first few rows of the dataframe
print(df.head())



**Step 3: Encode Labels**

*Explanation:*
We need to convert the text labels of sentiments (positive, neutral, negative) into numerical labels that the model can understand. This is done using LabelEncoder from sklearn.


In [ ]:
# Encode the sentiment labels to numerical values
le = LabelEncoder()
df['Sentiment'] = le.fit_transform(df['Sentiment'])

# Display the first few rows to see the changes
print(df.head())

**Step 4: Split the Dataset**

*Explanation:*
Splitting the dataset into training and validation sets helps in evaluating the model's performance on unseen data.

In [ ]:
# Split the dataset into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Text'], df['Sentiment'], test_size=0.2, random_state=42)

print(f'Training samples: {len(train_texts)}')
print(f'Validation samples: {len(val_texts)}')


**Step 5: Tokenize the Text Data**

*Explanation:*
Tokenization is the process of converting text into tokens that the model can process. We use BERT's tokenizer for this purpose.



In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the text data
def tokenize(texts):
    return tokenizer(texts.tolist(), padding=True, truncation=True, return_tensors="pt")

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)

print(f'Tokenized training data shape: {train_encodings["input_ids"].shape}')

**Step 6: Create PyTorch Dataset**

*Explanation:*
We wrap our tokenized data into a PyTorch Dataset class, which allows easy iteration over the data during training.


In [ ]:

class CustomerFeedbackDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create dataset objects
train_dataset = CustomerFeedbackDataset(train_encodings, train_labels)
val_dataset = CustomerFeedbackDataset(val_encodings, val_labels)


### Real-World Project: Part 2 - Model Training

**Step 7: Define the Model**

*Explanation:*
We define our sentiment classifier model using BERT as the base. The model includes a dropout layer and a fully connected layer to classify the sentiments.



In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, num_labels):
        super(SentimentClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1]
        output = self.dropout(pooled_output)
        return self.out(output)

# Instantiate the model
model = SentimentClassifier(num_labels=3)
print(model)


**Step 8: Set Up Training Parameters**

*Explanation:*
Define the loss function, optimizer, and move the model to the appropriate device (CPU or GPU).


In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# Check if GPU is available and move model to GPU if available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

**Step 9: Implement the Training Loop**

*Explanation:*
The training loop involves passing the data through the model, computing the loss, backpropagating the errors, and updating the weights.


In [ ]:
def train_model(model, train_dataset, val_dataset, epochs=3):
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1}, Training Loss: {total_loss/len(train_loader)}')

        model.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
        print(f'Epoch {epoch+1}, Validation Loss: {total_loss/len(val_loader)}')

train_model(model, train_dataset, val_dataset, epochs=3)


### Real-World Project: Part 3 - Model Evaluation

**Step 10: Define the Evaluation Function**

*Explanation:*
Evaluate the trained model on the validation dataset and generate a classification report.


In [ ]:
from sklearn.metrics import classification_report

def evaluate_model(model, val_dataset):
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    print(classification_report(true_labels, predictions, target_names=le.classes_))

evaluate_model(model, val_dataset)

### Final Project Walkthrough

**Overview of the Completed Project**

1. **Loading and Preprocessing the Dataset:**
   - Loaded the customer feedback dataset from the provided file.
   - Encoded the sentiments and split the dataset into training and validation sets.
   - Tokenized the text data using BERT tokenizer.

2. **Model Training:**
   - Defined a sentiment classifier model using BERT.
   - Implemented the training loop with appropriate loss function and optimizer.
   - Trained the model on the customer feedback dataset.

3. **Model Evaluation:**
   - Evaluated the trained model on the validation set.
   - Generated a classification report to assess the model's performance.

**Discussion of Key Takeaways and Best Practices**

- **Transfer Learning:** Leveraging pre-trained models like BERT can significantly improve performance on NLP tasks with limited data.
- **Data Preprocessing:** Properly encoding labels and tokenizing text are crucial steps in preparing the dataset.
- **Model Evaluation:** Always validate your model on a separate dataset to ensure it generalizes well to new data.
- **Continuous Improvement:** Based on evaluation results, fine-tuning the model and hyperparameters can lead to better performance